In [0]:
# Bronze — Source 16: cloudwatch
import sys
sys.path.append('/Workspace/Users/sutharripal26@gmail.com/ecommerce-lakehouse/pipelines/bronze/shared')
from bronze_utils import get_watermark, update_watermark

RAW_BUCKET = 's3://ecommerce-lakehouse-467091806172-raw-01'
SOURCE = '16_cloudwatch'
TARGET_TABLE = 'bronze.src_16_cloudwatch.app_logs'
MERGE_KEY = 'event_id'
PATH = 's3://ecommerce-lakehouse-467091806172-raw-01/source=16_cloudwatch/app.logs/year=*/month=*/day=*/'


In [0]:
from pyspark.sql.functions import col, lit, max as spark_max, from_json
from pyspark.sql.types import *
import json as json_lib

watermark = get_watermark(spark, SOURCE)
print(f'[{SOURCE}] Watermark: {watermark}')

raw_df = spark.read.text(PATH) \
    .filter(col('_metadata.file_modification_time') > lit(watermark))

if raw_df.count() == 0:
    print(f'[{SOURCE}] No new files — skipping')
    dbutils.notebook.exit('No new data')

from pyspark.sql.functions import udf

@udf(returnType=StringType())
def unwrap_json(s):
    if s is None: return None
    try:
        inner = json_lib.loads(s)
        if isinstance(inner, str): return inner
        import json as j; return j.dumps(inner)
    except: return s

schema = StructType([
    StructField('event_id', StringType()),
    StructField('timestamp', StringType()),
    StructField('log_level', StringType()),
    StructField('service', StringType()),
    StructField('message', StringType()),
    StructField('request_id', StringType()),
    StructField('user_id', LongType()),
    StructField('order_id', LongType()),
    StructField('error_code', StringType()),
    StructField('duration_ms', LongType()),
    StructField('environment', StringType()),
])

df = raw_df \
    .withColumn('unwrapped', unwrap_json(col('value'))) \
    .withColumn('parsed', from_json(col('unwrapped'), schema)) \
    .select('parsed.*') \
    .filter(col(MERGE_KEY).isNotNull())

row_count = df.count()
print(f'[{SOURCE}] {row_count} rows parsed')

spark.sql('CREATE SCHEMA IF NOT EXISTS bronze.src_16_cloudwatch')

if spark.catalog.tableExists(TARGET_TABLE):
    from delta.tables import DeltaTable
    dt = DeltaTable.forName(spark, TARGET_TABLE)
    dt.alias('t').merge(df.alias('s'), f't.{MERGE_KEY} = s.{MERGE_KEY}') \
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print('MERGE complete')
else:
    df.write.format('delta').mode('overwrite') \
        .option('mergeSchema', 'true').saveAsTable(TARGET_TABLE)
    print('Initial load complete')

latest_ts = raw_df.select(spark_max('_metadata.file_modification_time')).collect()[0][0]
update_watermark(spark, SOURCE, latest_ts, row_count)
print(f'Watermark updated to {latest_ts}')


In [0]:
count = spark.sql(f'SELECT COUNT(*) as cnt FROM {TARGET_TABLE}').collect()[0]['cnt']
print(f'{TARGET_TABLE}: {count} rows')
spark.sql(f"SELECT * FROM bronze.pipeline.watermarks WHERE source = '{SOURCE}'").show()
